In [1]:
%pip install -q pandas sqlalchemy matplotlib


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, inspect, text
from IPython.display import display

In [3]:
products = pd.DataFrame([
    [1, "Keyboard", "Accessories", 15000],
    [2, "Mouse", "Accessories", 8000],
    [3, "Monitor", "Displays", 90000],
    [4, "SSD", "Storage", 30000],
], columns=["product_id", "product", "category", "unit_price"])

sales = pd.DataFrame([
    [101, 1, 2, "Almaty", "completed"],
    [102, 2, 3, "Astana", "completed"],
    [103, 3, 1, "Almaty", "completed"],
    [104, 4, 2, "Astana", "completed"],
    [105, 1, None, "Almaty", "completed"],
    [106, 2, 1, None, "completed"],
    [107, 3, 1, "Astana", "cancelled"],
    [108, 4, 0, "Almaty", "completed"],
    [109, 1, 1, "Astana", "completed"],
    [110, 4, 3, "Almaty", "completed"],
    [111, 2, 2, "Almaty", "completed"],
    [101, 1, 2, "Almaty", "completed"],
], columns=["sale_id", "product_id", "quantity", "city", "status"])

products.to_csv("products.csv", index=False)
sales.to_csv("sales_raw.csv", index=False)
sales


,sale_id,product_id,quantity,city,status
0,101,1,2.0,Almaty,completed
1,102,2,3.0,Astana,completed
2,103,3,1.0,Almaty,completed
3,104,4,2.0,Astana,completed
4,105,1,NaN,Almaty,completed
5,106,2,1.0,None,completed
6,107,3,1.0,Astana,cancelled
7,108,4,0.0,Almaty,completed
8,109,1,1.0,Astana,completed
9,110,4,3.0,Almaty,completed


In [4]:
engine = create_engine("sqlite:///lab6_shop.db")
products.to_sql("products", engine, if_exists="replace", index=False)
sales.to_sql("sales", engine, if_exists="replace", index=False)

print("Tables:", inspect(engine).get_table_names())
with engine.connect() as connection:
    sales_df = pd.read_sql(text("SELECT * FROM sales"), connection)

print("Rows, columns:", sales_df.shape)
print("Column names:", sales_df.columns.tolist())
print("Data types:")
print(sales_df.dtypes)
display(sales_df.head())

Tables: ['products', 'sales']
Rows, columns: (12, 5)
Column names: ['sale_id', 'product_id', 'quantity', 'city', 'status']
Data types:
sale_id         int64
product_id      int64
quantity      float64
city           object
status         object
dtype: object


,sale_id,product_id,quantity,city,status
0,101,1,2.0,Almaty,completed
1,102,2,3.0,Astana,completed
2,103,3,1.0,Almaty,completed
3,104,4,2.0,Astana,completed
4,105,1,NaN,Almaty,completed


In [5]:
with engine.connect() as connection:
    filtered_df = pd.read_sql(text("""
        SELECT * FROM sales
        WHERE status = 'completed' AND quantity >= 2
    """), connection)
    sorted_df = pd.read_sql(text("""
        SELECT * FROM products ORDER BY unit_price DESC
    """), connection)
    grouped_df = pd.read_sql(text("""
        SELECT city, SUM(quantity) AS total_quantity
        FROM sales WHERE status = 'completed'
        GROUP BY city
    """), connection)
    joined_df = pd.read_sql(text("""
        SELECT s.sale_id, s.product_id, p.product, p.category,
               s.quantity, p.unit_price, s.city, s.status
        FROM sales AS s
        JOIN products AS p ON s.product_id = p.product_id
    """), connection)

print("Filtered sales")
display(filtered_df)
print("Products sorted by price")
display(sorted_df)
print("Raw quantity totals by city")
display(grouped_df)
print("Joined data")
display(joined_df.head())

Filtered sales


,sale_id,product_id,quantity,city,status
0,101,1,2.0,Almaty,completed
1,102,2,3.0,Astana,completed
2,104,4,2.0,Astana,completed
3,110,4,3.0,Almaty,completed
4,111,2,2.0,Almaty,completed
5,101,1,2.0,Almaty,completed


Products sorted by price


,product_id,product,category,unit_price
0,3,Monitor,Displays,90000
1,4,SSD,Storage,30000
2,1,Keyboard,Accessories,15000
3,2,Mouse,Accessories,8000


Raw quantity totals by city


,city,total_quantity
0,None,1.0
1,Almaty,10.0
2,Astana,6.0


Joined data


,sale_id,product_id,product,category,quantity,unit_price,city,status
0,101,1,Keyboard,Accessories,2.0,15000,Almaty,completed
1,102,2,Mouse,Accessories,3.0,8000,Astana,completed
2,103,3,Monitor,Displays,1.0,90000,Almaty,completed
3,104,4,SSD,Storage,2.0,30000,Astana,completed
4,105,1,Keyboard,Accessories,NaN,15000,Almaty,completed


In [6]:
df = joined_df.copy()
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df = df.dropna(subset=["quantity"]).copy()
df["city"] = df["city"].fillna("Unknown")
df = df.drop_duplicates(subset=["sale_id"])
df = df[(df["status"] == "completed") & (df["quantity"] > 0)].copy()
df["quantity"] = df["quantity"].astype(int)
df["revenue"] = df["quantity"] * df["unit_price"]
df = df.drop(columns=["status"]).reset_index(drop=True)
df

,sale_id,product_id,product,category,quantity,unit_price,city,revenue
0,101,1,Keyboard,Accessories,2,15000,Almaty,30000
1,102,2,Mouse,Accessories,3,8000,Astana,24000
2,103,3,Monitor,Displays,1,90000,Almaty,90000
3,104,4,SSD,Storage,2,30000,Astana,60000
4,106,2,Mouse,Accessories,1,8000,Unknown,8000
5,109,1,Keyboard,Accessories,1,15000,Astana,15000
6,110,4,SSD,Storage,3,30000,Almaty,90000
7,111,2,Mouse,Accessories,2,8000,Almaty,16000


In [7]:
%%writefile lab6_etl.py
import pandas as pd
from sqlalchemy import create_engine, text
sales = pd.read_csv("sales_raw.csv")
products = pd.read_csv("products.csv")
clean = sales[["sale_id", "product_id", "quantity", "city", "status"]].copy()
clean["quantity"] = pd.to_numeric(clean["quantity"], errors="coerce")
clean = clean.dropna(subset=["quantity"]).copy()
clean["city"] = clean["city"].fillna("Unknown")
clean = clean.drop_duplicates(subset=["sale_id"])
clean = clean[(clean["status"] == "completed") & (clean["quantity"] > 0)].copy()
clean["quantity"] = clean["quantity"].astype(int)
clean = clean.merge(products, on="product_id", how="left", validate="many_to_one")
assert clean["unit_price"].notna().all(), "A product ID is missing from products.csv"
clean["revenue"] = clean["quantity"] * clean["unit_price"]
clean = clean.drop(columns=["status"])
engine = create_engine("sqlite:///lab6_shop.db")
clean.to_sql("clean_sales", engine, if_exists="replace", index=False)
with engine.connect() as connection:
    saved = pd.read_sql(text("SELECT * FROM clean_sales ORDER BY sale_id"), connection)
print("Saved records:", len(saved))
print(saved.head().to_string(index=False))


Writing lab6_etl.py


In [8]:
%run lab6_etl.py

Saved records: 8
 sale_id  product_id  quantity    city  product    category  unit_price  revenue
     101           1         2  Almaty Keyboard Accessories       15000    30000
     102           2         3  Astana    Mouse Accessories        8000    24000
     103           3         1  Almaty  Monitor    Displays       90000    90000
     104           4         2  Astana      SSD     Storage       30000    60000
     106           2         1 Unknown    Mouse Accessories        8000     8000


In [9]:
source = pd.read_csv("sales_raw.csv")
with engine.connect() as connection:
    final_df = pd.read_sql(text("SELECT * FROM clean_sales ORDER BY sale_id"), connection)

comparison = pd.DataFrame({
    "Check": ["Rows", "Missing cells", "Duplicate sale IDs"],
    "Source": [len(source), source.isna().sum().sum(), source.duplicated("sale_id").sum()],
    "Final": [len(final_df), final_df.isna().sum().sum(), final_df.duplicated("sale_id").sum()]
})
display(comparison)
display(pd.DataFrame({"Source type": source.dtypes.astype(str),
                      "Final type": final_df.dtypes.astype(str)}).fillna("Not present"))

expected = df.sort_values("sale_id").reset_index(drop=True)
pd.testing.assert_frame_equal(final_df[expected.columns], expected)
assert final_df.isna().sum().sum() == 0
assert final_df["sale_id"].is_unique
assert (final_df["quantity"] > 0).all()
assert (final_df["revenue"] == final_df["quantity"] * final_df["unit_price"]).all()
print("All checks passed")

,Check,Source,Final
0,Rows,12,8
1,Missing cells,2,0
2,Duplicate sale IDs,1,0


,Source type,Final type
category,Not present,object
city,object,object
product,Not present,object
product_id,int64,int64
quantity,float64,int64
revenue,Not present,int64
sale_id,int64,int64
status,object,Not present
unit_price,Not present,int64


All checks passed


In [11]:
findings = [
    f"The ETL reduced {len(source)} source rows to {len(final_df)} valid sales: one duplicate, one missing quantity, one cancelled sale and one zero quantity were removed.",
    f"Missing cells decreased from {source.isna().sum().sum()} to {final_df.isna().sum().sum()}. The missing quantity row was dropped; the missing city became Unknown.",
    f"Duplicate sale IDs decreased from {source.duplicated('sale_id').sum()} to {final_df.duplicated('sale_id').sum()}.",
    f"Quantity changed from {source['quantity'].dtype} to {final_df['quantity'].dtype}; all retained quantities are positive integers.",
    f"Total revenue is {final_df['revenue'].sum():,.0f} KZT. {category_totals.idxmax()} has the highest category revenue ({category_totals.max():,.0f} KZT). The saved data matches the cleaned query result."
]
for number, finding in enumerate(findings, 1):
    print(f"{number}. {finding}")

1. The ETL reduced 12 source rows to 8 valid sales: one duplicate, one missing quantity, one cancelled sale and one zero quantity were removed.
2. Missing cells decreased from 2 to 0. The missing quantity row was dropped; the missing city became Unknown.
3. Duplicate sale IDs decreased from 1 to 0.
4. Quantity changed from float64 to int64; all retained quantities are positive integers.
5. Total revenue is 333,000 KZT. Storage has the highest category revenue (150,000 KZT). The saved data matches the cleaned query result.
